# 00 — Validacija izvora podataka (Phase 0)

Reproduces the Phase 0 checks from `PLAN.md`: does every planned source answer for a point in Zagreb?
Results and decisions are recorded in `docs/data-sources.md`. All requests are read-only; nothing is written to disk.

Coordinates are EPSG:4326 (lat, lon) unless stated; the air-index WFS returns EPSG:3765 (HTRS96/TM).

In [1]:
import io, math, xml.etree.ElementTree as ET
import httpx
from PIL import Image
from pyproj import Transformer

HEADERS = {'User-Agent': 'EkoAdresa/0.1 (Datathon 2026 Phase 0 validation)'}
client = httpx.Client(headers=HEADERS, timeout=60, follow_redirects=True,
                      transport=httpx.HTTPTransport(retries=3))

POINTS = {  # name: (lat, lon), EPSG:4326
    'Savica': (45.7905, 15.9950),
    'Trg bana Jelačića': (45.8131, 15.9772),
    'Sljeme': (45.8986, 15.9486),
}

import time

def get(url, params=None, attempts=4):
    # servisi.voda.hr intermittently answers 502 -> retry 5xx with backoff
    for i in range(attempts):
        r = client.get(url, params=params)
        if r.status_code < 500 or i == attempts - 1:
            r.raise_for_status()
            return r
        print(f'  retry {i + 1} after HTTP {r.status_code}')
        time.sleep(2 ** i)

def distance_m(lat1, lon1, lat2, lon2):
    # Haversine
    p1, p2 = math.radians(lat1), math.radians(lat2)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(math.radians(lon2 - lon1) / 2) ** 2
    return 2 * 6_371_000 * math.asin(math.sqrt(a))

## 1. Poplave — `hv-poplave` (Hrvatske vode, WMS)

GetCapabilities, then the GetFeatureInfo attempt (expected: `LayerNotQueryable`), then the GetMap centre-pixel workaround.

In [2]:
HV = 'https://servisi.voda.hr/poplave_opasnosti/ows'
caps = ET.fromstring(get(HV, {'SERVICE': 'WMS', 'REQUEST': 'GetCapabilities'}).content)
W = '{http://www.opengis.net/wms}'
for layer in caps.iter(W + 'Layer'):
    name = layer.find(W + 'Name')
    if name is not None:
        scales = [e.text for e in layer if e.tag.endswith('ScaleDenominator')]
        print(layer.get('queryable'), name.text, scales)

0 hr.fd.opasnost-od-poplave-mala-vjerojatnost ['20000.0', '5000000.0']
0 hr.fd.opasnost-od-poplave-srednja-vjerojatnost ['20000.0', '5000000.0']
0 hr.fd.opasnost-od-poplave-velika-vjerojatnost ['20000.0', '5000000.0']
0 hr.fd.vjerojatnost-pojavljivanja ['20000.0', '5000000.0']


In [3]:
LAYER = 'hr.fd.opasnost-od-poplave-srednja-vjerojatnost'
lat, lon = POINTS['Savica']
r = client.get(HV, params={'SERVICE': 'WMS', 'VERSION': '1.3.0', 'REQUEST': 'GetFeatureInfo',
    'LAYERS': LAYER, 'QUERY_LAYERS': LAYER, 'STYLES': '', 'CRS': 'CRS:84',
    'BBOX': f'{lon-0.0005},{lat-0.0005},{lon+0.0005},{lat+0.0005}', 'WIDTH': 101, 'HEIGHT': 101,
    'I': 50, 'J': 50, 'INFO_FORMAT': 'application/json'})
print('LayerNotQueryable' in r.text, r.text[:250])

True <?xml version="1.0" encoding="UTF-8"?><ServiceExceptionReport version="1.3.0" xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc https://servisi.voda.hr/schemas/wms/


In [4]:
def flood_pixel(scenario, lat, lon, half_m=353.5, px=101):
    """Centre pixel of a ~707 m GetMap (~7 m/px, ~1:25 000; layers do not draw below 1:20 000)."""
    dlon = half_m / (111_320 * math.cos(math.radians(lat)))
    dlat = half_m / 110_574
    r = get(HV, {'SERVICE': 'WMS', 'VERSION': '1.3.0', 'REQUEST': 'GetMap',
        'LAYERS': f'hr.fd.opasnost-od-poplave-{scenario}-vjerojatnost', 'STYLES': '', 'CRS': 'CRS:84',
        'BBOX': f'{lon-dlon},{lat-dlat},{lon+dlon},{lat+dlat}', 'WIDTH': px, 'HEIGHT': px,
        'FORMAT': 'image/png', 'TRANSPARENT': 'TRUE'})
    rgba = Image.open(io.BytesIO(r.content)).convert('RGBA').getpixel((px // 2, px // 2))
    return None if rgba[3] == 0 else '#%02X%02X%02X' % rgba[:3]

for name, (lat, lon) in POINTS.items():
    print(name, {s: flood_pixel(s, lat, lon) for s in ('velika', 'srednja', 'mala')})

  retry 1 after HTTP 502


  retry 2 after HTTP 502


  retry 1 after HTTP 502


  retry 2 after HTTP 502


Savica {'velika': None, 'srednja': None, 'mala': '#73B2FF'}


Trg bana Jelačića {'velika': None, 'srednja': None, 'mala': None}
Sljeme {'velika': None, 'srednja': None, 'mala': None}


In [5]:
legend = get(HV, {'service': 'WMS', 'version': '1.3.0', 'request': 'GetLegendGraphic',
                  'format': 'application/json', 'layer': LAYER}).json()
for rule in legend['Legend'][0]['rules']:
    print(rule['title'], rule.get('filter'), rule['symbolizers'][0]['Polygon'].get('fill'))

Dubina < 0,5 m [m_kl_dub = '1'] #BEE8FF
Dubina 0,5 m - 1,5 m [m_kl_dub = '2'] #73DFFF
Dubina 1,5 m - 2,5 m [m_kl_dub = '3'] #73B2FF
Dubina > 2,5 m [m_kl_dub = '4'] #7A8EF5


## 2. Buka — `zg-buka` (Grad Zagreb, WMS GetFeatureInfo)

In [6]:
ZG = 'https://geoportal.zagreb.hr/Public/StrategijskoPlaniranje_Public/MapServer/WMSServer'
NOISE_LAYERS = ','.join(f'SKB_R4_{s}_{i}'
    for s in ('Cestovni_promet', 'Tramvajski_promet', 'Zeljeznicki_promet', 'Industrijski_pogon')
    for i in ('Lden', 'Lnight'))

def noise(lat, lon):
    d = 0.0001
    r = get(ZG, {'SERVICE': 'WMS', 'VERSION': '1.3.0', 'REQUEST': 'GetFeatureInfo',
        'LAYERS': NOISE_LAYERS, 'QUERY_LAYERS': NOISE_LAYERS, 'STYLES': '', 'CRS': 'CRS:84',
        'BBOX': f'{lon-d},{lat-d},{lon+d},{lat+d}', 'WIDTH': 101, 'HEIGHT': 101, 'I': 50, 'J': 50,
        'INFO_FORMAT': 'application/json', 'FEATURE_COUNT': 20, 'BUFFER': 0})
    return {f['id'].split('.')[0].removeprefix('SKB_R4_'): f['properties']['razina'] for f in r.json()['features']}

for name, (lat, lon) in POINTS.items():
    print(name, noise(lat, lon) or 'no features')

Savica {'Cestovni_promet_Lden': 70, 'Cestovni_promet_Lnight': 60}


Trg bana Jelačića {'Cestovni_promet_Lden': 50, 'Tramvajski_promet_Lden': 70, 'Tramvajski_promet_Lnight': 60}
Sljeme no features


## 3. Zrak — stations (`zg-zrak-postaje`), 2023 measurements (`zg-zrak-mjerenja`), current index (`rh-zrak-indeks`)

In [7]:
stations = get('https://opendata.arcgis.com/api/v3/datasets/ed9898e63ec94770a8fb37713c36f8ca_0/downloads/data',
               {'format': 'geojson', 'spatialRefId': 4326, 'where': '1=1'}).json()['features']
for f in stations:
    lon, lat = f['geometry']['coordinates']
    print(f"{f['properties']['naziv']:<28} {f['properties']['tip_postaje']:<12} {lat:.5f}, {lon:.5f}")

Đorđićeva ulica              Prometna     45.81077, 15.97945
Ksaverska cesta              Pozadinska   45.83461, 15.97869
Peščenica                    Industrijska 45.80425, 16.02824
Prilaz baruna Filipovića     Prometna     45.81135, 15.94454
Siget                        Prometna     45.77340, 15.97922
Susedgrad                    Industrijska 45.81158, 15.86887


In [8]:
import openpyxl
url = ('https://data.zagreb.hr/dataset/715ef845-ca2d-429e-8c08-8de63ad7d4e9/resource/'
       'e3a785e7-fdfa-48b0-b29b-638c57acd5c9/download/djordjiceva20237.xlsx')
ws = openpyxl.load_workbook(io.BytesIO(get(url).content), read_only=True, data_only=True)['Numeričke vrijednosti']
rows = list(ws.iter_rows(values_only=True))
print('header:', rows[0])
data = [r for r in rows[1:] if r[0] is not None]
print('days:', len(data), data[0][0].date(), '->', data[-1][0].date())
print('first row:', data[0])

header: ('Datum', 'NO2 [µg/m3]', 'Ozon  [µg/m3]', 'PM10 [µg/m3]', 'Pb u PM10 [µg/m3]', 'Mn u PM10 [µg/m3]', 'Cd u PM10 [ng/m3]', 'Ni u PM10 [ng/m3]', 'As u PM10 [ng/m3]', None)
days: 31 2023-07-01 -> 2023-07-31
first row: (datetime.datetime(2023, 7, 1, 0, 0), 34.79663037952796, 38.795931303964664, 14.531930333817057, None, None, None, None, None, None)


In [9]:
AQ = 'https://servisi.azo.hr/inspire/geos1/hh/wfs'
ns = {'hh1': 'http://inspire.ec.europa.eu/schemas/hh/4.0', 'gml': 'http://www.opengis.net/gml/3.2'}
to_wgs84 = Transformer.from_crs('EPSG:3765', 'EPSG:4326', always_xy=True)  # HTRS96/TM -> WGS84
fc = ET.fromstring(get(AQ, {'service': 'WFS', 'version': '2.0.0', 'request': 'GetFeature',
                            'typeNames': 'hh1:EnvHealthDeterminantMeasure'}).content)
measures = fc.findall('.//hh1:EnvHealthDeterminantMeasure', ns)
print('features in Croatia:', len(measures))
for m in measures:
    x, y = map(float, m.find('.//gml:pos', ns).text.split())
    lon, lat = to_wgs84.transform(x, y)
    if 45.70 < lat < 45.92 and 15.80 < lon < 16.25:  # Zagreb bbox
        v = m.find('hh1:measure', ns)
        print(m.get('{http://www.opengis.net/gml/3.2}id'), f'{lat:.5f}, {lon:.5f}',
              m.find('.//gml:endPosition', ns).text, v.get('uom'), v.text)

features in Croatia: 25
EnvHealthDeterminantMeasure.155 45.80034, 15.97407 2026-11-04T06:00:00Z AQI 1.0
EnvHealthDeterminantMeasure.157 45.76495, 16.00647 2026-11-04T06:00:00Z AQI 5.0


## 4. Otpad — recycling yards (`zg-reciklaza`) and green islands (`zg-zeleni-otoci`)

In [10]:
yards = get('https://opendata.arcgis.com/api/v3/datasets/249fa384ccf9481abf4fd2de73a822f5_0/downloads/data',
            {'format': 'geojson', 'spatialRefId': 4326, 'where': '1=1'}).json()['features']
print('yards:', len(yards))
for name, (lat, lon) in POINTS.items():
    d, y = min((distance_m(lat, lon, f['geometry']['coordinates'][1], f['geometry']['coordinates'][0]), f) for f in yards)
    print(f"{name}: nearest {y['properties']['NAZIV']} at {d:.0f} m (straight line)")

yards: 20
Savica: nearest MRD TRNJE at 1286 m (straight line)
Trg bana Jelačića: nearest MRD GORNJI GRAD - MEDVEŠČAK at 1693 m (straight line)
Sljeme: nearest MRD PODSLJEME, KRALJIČIN ZDENAC at 4824 m (straight line)


In [11]:
raw = get('https://data.zagreb.hr/dataset/0b532fd6-fa24-4f90-8c5e-698b5cbe94b7/resource/'
          '5d15eae4-6aa7-426c-b02c-a8fdb3d61a74/download/zeleniotoci.csv').content.decode('cp1250')
lines = raw.splitlines()
print('header:', lines[0]); print('rows:', len(lines) - 1); print(lines[1:4])
print('-> no coordinate columns: geocoding or another source needed')

header: ULICA;LOKACIJA (OPIS);GRADSKA ČETVRT
rows: 297
['Sesvetska Sopnica, Prelčeva 20;uz Prelčevu ulicu;SESVETE', 'Luka,Bistrička 1;uz prilaz parkirališta i objekta Vodoprivrede;SESVETE', 'Križanje Potočnica-Narcisa;sjeverna strana;SESVETE']
-> no coordinate columns: geocoding or another source needed


## 5. Zelena mobilnost — ZET bus and tram stops (`zg-zet`)

In [12]:
bus = get('https://data.zagreb.hr/dataset/cc7e45ae-4bad-4a07-bca2-25e82f1e082c/resource/'
          '3b12a182-afa3-4b07-833f-17d8bcf80ea9/download/data.geojson').json()['features']
tram = get('https://opendata.arcgis.com/api/v3/datasets/8f84d42e684141eaa7ceb26d9c9b8f5a_0/downloads/data',
           {'format': 'geojson', 'spatialRefId': 4326, 'where': '1=1'}).json()['features']
print('bus platforms:', len(bus), '| tram platforms:', len(tram))
for name, (lat, lon) in POINTS.items():
    for kind, fs in (('bus', bus), ('tram', tram)):
        ds = [distance_m(lat, lon, f['geometry']['coordinates'][1], f['geometry']['coordinates'][0]) for f in fs]
        print(f'{name:<18} {kind:<4} <=300 m: {sum(d <= 300 for d in ds):>2}  <=500 m: {sum(d <= 500 for d in ds):>2}  nearest: {min(ds):.0f} m')

bus platforms: 1888 | tram platforms: 260
Savica             bus  <=300 m:  2  <=500 m:  7  nearest: 57 m
Savica             tram <=300 m:  0  <=500 m:  0  nearest: 592 m
Trg bana Jelačića  bus  <=300 m:  4  <=500 m: 10  nearest: 100 m
Trg bana Jelačića  tram <=300 m:  2  <=500 m:  6  nearest: 27 m
Sljeme             bus  <=300 m:  1  <=500 m:  3  nearest: 145 m
Sljeme             tram <=300 m:  0  <=500 m:  0  nearest: 4897 m


## Summary

See `docs/data-sources.md` → *Go / no-go per layer*.